# Tutorial: Quality Control of CMORised Output

This notebook demonstrates how to use the `access_moppy.qc` framework to run
Quality Control checks on CMORised ACCESS-ESM output.

The framework is designed to be **separate from the CMORisation pipeline**,
so you can run checks:
- immediately after CMORising a variable (with full CMOR table context), or
- later on any existing CMORised NetCDF file.

The checks cover two tiers derived from the ESM1.6 pre-publication QC document:

| Tier | Examples |
|---|---|
| **Technical** | Required CMIP6 attributes, units, cell_methods, time monotonicity, lat/lon ranges |
| **Science** | Global min/max within physical bounds, non-negative variables, global mean plausibility |

## 1. Imports

In [ ]:
import numpy as np
import xarray as xr

from access_moppy.qc import QCRunner, QCReport, QCStatus, get_checks

## 2. Running QC on an existing CMORised file

The simplest use case — open a file and run all registered checks.

The only required context key is `cmor_name` (the CMIP variable short name).
Checks that need the CMOR table (e.g. units verification, valid_min/valid_max)
will silently `SKIP` if no vocabulary object is provided.

In [ ]:
# Replace with the path to your CMORised file
# ds = xr.open_dataset("tas_Amon_ACCESS-ESM1-5_historical_r1i1p1f1_gn_185001-201412.nc")

# --- Synthetic dataset for demonstration ---
time = np.arange(120, dtype="float64")  # 120 monthly steps
lat = np.linspace(-90, 90, 73)
lon = np.linspace(0, 358.125, 192, endpoint=True)
data = np.random.uniform(272, 302, (120, 73, 192)).astype("float32")

lat_bnds = np.column_stack([lat - 1.25, lat + 1.25])
lon_bnds = np.column_stack([lon - 0.9375, lon + 0.9375])
time_bnds = np.column_stack([time - 0.5, time + 0.5])

ds = xr.Dataset(
    {
        "tas": xr.DataArray(
            data,
            dims=["time", "lat", "lon"],
            attrs={
                "units": "K",
                "cell_methods": "area: time: mean",
                "_FillValue": 1e20,
                "missing_value": 1e20,
                "long_name": "Near-Surface Air Temperature",
            },
        ),
        "time_bnds": xr.DataArray(time_bnds, dims=["time", "bnds"]),
        "lat_bnds": xr.DataArray(lat_bnds, dims=["lat", "bnds"]),
        "lon_bnds": xr.DataArray(lon_bnds, dims=["lon", "bnds"]),
    },
    coords={"time": time, "lat": lat, "lon": lon},
    attrs={
        "variable_id": "tas",
        "table_id": "Amon",
        "activity_id": "CMIP",
        "experiment_id": "historical",
        "source_id": "ACCESS-ESM1-5",
        "variant_label": "r1i1p1f1",
        "grid_label": "gn",
        "frequency": "mon",
        "realm": "atmos",
        "institution_id": "CSIRO-ARCCSS",
        "mip_era": "CMIP6",
        "Conventions": "CF-1.7 CMIP-6.2",
        "creation_date": "2026-06-09T00:00:00Z",
    },
)

ds

In [ ]:
runner = QCRunner()
report = runner.run(ds, context={"cmor_name": "tas"})
print(report.summary())

## 3. Running QC with full CMOR table context

When you pass a `vocab` object in the context, additional checks become active:
- **units** verification against the CMOR table
- **cell_methods** verification
- **valid_min / valid_max** bounds from the CMOR table

This is the recommended way to run QC immediately after CMORisation,
when the vocabulary object is already available.

In [ ]:
from access_moppy.vocabulary_processors import CMIP6Vocabulary

vocab = CMIP6Vocabulary(
    compound_name="Amon.tas",
    experiment_id="historical",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
)

report_with_vocab = QCRunner().run(
    ds,
    context={"cmor_name": "tas", "vocab": vocab},
)
print(report_with_vocab.summary())

## 4. Exploring the report

The `QCReport` object gives you structured access to results.

In [ ]:
report_with_vocab

In [ ]:
# Overall status
print("Overall:", report_with_vocab.overall_status.value)

# Failures
print("\n--- Failures ---")
for r in report_with_vocab.failures:
    print(f"  {r.check_name}: {r.message}")
    if r.details:
        for k, v in r.details.items():
            print(f"    {k}: {v}")

# Warnings
print("\n--- Warnings ---")
for r in report_with_vocab.warnings:
    print(f"  {r.check_name}: {r.message}")

# Skipped checks
print("\n--- Skipped ---")
for r in report_with_vocab.skipped:
    print(f"  {r.check_name}: {r.message}")

In [ ]:
# Export to JSON for automated pipelines or logging
json_str = report_with_vocab.to_json()  # pass a file path to write to disk
print(json_str[:800], "...")

## 5. Deliberately broken dataset — seeing FAILs

Let's construct a dataset with several deliberate errors to see what the
checks catch.

In [ ]:
# Dataset with problems:
#  1. Negative precipitation values
#  2. Non-monotonic time
#  3. Wrong units
#  4. Missing required attribute (institution_id)
#  5. _FillValue != missing_value
time_bad = np.array([0, 1, 2, 4, 3, 5], dtype="float64")  # non-monotonic
pr_data = np.random.uniform(-0.001, 0.002, (6, 10, 10)).astype("float32")  # negative values

ds_broken = xr.Dataset(
    {
        "pr": xr.DataArray(
            pr_data,
            dims=["time", "lat", "lon"],
            attrs={
                "units": "mm/day",          # wrong — CMIP expects kg m-2 s-1
                "cell_methods": "area: mean",
                "_FillValue": 1e20,
                "missing_value": 9.96921e+36,  # inconsistent
            },
        ),
    },
    coords={
        "time": time_bad,
        "lat": np.linspace(-90, 90, 10),
        "lon": np.linspace(0, 360, 10, endpoint=False),
    },
    attrs={
        "variable_id": "pr",
        "table_id": "Amon",
        "activity_id": "CMIP",
        "experiment_id": "historical",
        "source_id": "ACCESS-ESM1-5",
        "variant_label": "r1i1p1f1",
        "grid_label": "gn",
        "frequency": "mon",
        "realm": "atmos",
        # institution_id deliberately omitted
        "mip_era": "CMIP6",
        "Conventions": "CF-1.7 CMIP-6.2",
        "creation_date": "2026-06-09T00:00:00Z",
    },
)

vocab_pr = CMIP6Vocabulary(
    compound_name="Amon.pr",
    experiment_id="historical",
    source_id="ACCESS-ESM1-5",
    variant_label="r1i1p1f1",
    grid_label="gn",
    activity_id="CMIP",
)

report_broken = QCRunner().run(ds_broken, context={"cmor_name": "pr", "vocab": vocab_pr})
print(report_broken.summary())

## 6. Running only a subset of checks

Use `get_checks()` with explicit names to build a lightweight runner for a
specific task.

In [ ]:
# Run only the temporal checks
temporal_checks = get_checks([
    "temporal.time_monotonicity",
    "temporal.time_duplicates",
    "temporal.time_bounds_present",
])

temporal_runner = QCRunner(checks=temporal_checks)
report_temporal = temporal_runner.run(ds_broken, context={"cmor_name": "pr"})
print(report_temporal.summary())

In [ ]:
# Run only the science / global-stats checks
science_checks = get_checks([
    "global_stats.min_max_range",
    "global_stats.non_negative",
    "global_stats.global_mean_range",
])

science_runner = QCRunner(checks=science_checks)
report_science = science_runner.run(ds_broken, context={"cmor_name": "pr"})
print(report_science.summary())

## 7. Writing a custom check

Adding a check requires:
1. Subclass `QCCheck` and set a unique `name`.
2. Implement `run(ds, context) -> QCResult`.
3. Call `register_check(MyCheck())` — it then runs automatically with every `QCRunner()`.

Example: TOA energy balance check (from the QC document).
This is a multi-variable check, so it expects a dataset that already
has `rsdt`, `rsut`, and `rlut` merged together.

In [ ]:
from access_moppy.qc import QCCheck, register_check, QCRunner


class TOAEnergyBalanceCheck(QCCheck):
    """Global-mean TOA net radiation N = rsdt − rsut − rlut must be near zero.

    From the ESM1.6 QC document:
       'TOA energy balance: N = rsdt − rsut − rlut (mean imbalance, drift)'

    This check expects all three radiation variables to be present
    in the dataset. Pass a merged multi-variable dataset to use it.
    """

    name = "science.toa_energy_balance"
    _tolerance_wm2: float = 1.0  # W m-2; typical piControl tolerance

    def run(self, ds, context):
        required = {"rsdt", "rsut", "rlut"}
        missing = required - set(ds.data_vars)
        if missing:
            return self._skip(
                f"Requires {sorted(required)}; missing {sorted(missing)} "
                "— pass a merged dataset containing all three variables"
            )

        N = float((ds["rsdt"] - ds["rsut"] - ds["rlut"]).mean().values)
        if abs(N) > self._tolerance_wm2:
            return self._warn(
                f"TOA energy imbalance |N| = {N:.3f} W m-2 "
                f"(tolerance ±{self._tolerance_wm2} W m-2)",
                toa_imbalance_wm2=N,
                tolerance_wm2=self._tolerance_wm2,
            )
        return self._pass(
            f"TOA energy balance N = {N:.3f} W m-2",
            toa_imbalance_wm2=N,
        )


register_check(TOAEnergyBalanceCheck())
print("Custom check registered. All checks now:", [c.name for c in get_checks()])

In [ ]:
# Build a synthetic multi-variable radiation dataset to test the check
n = (60, 73, 192)
ds_rad = xr.Dataset(
    {
        "rsdt": xr.DataArray(
            np.random.uniform(200, 400, n).astype("float32"),
            dims=["time", "lat", "lon"],
            attrs={"units": "W m-2", "_FillValue": 1e20, "missing_value": 1e20},
        ),
        "rsut": xr.DataArray(
            np.random.uniform(50, 120, n).astype("float32"),
            dims=["time", "lat", "lon"],
            attrs={"units": "W m-2", "_FillValue": 1e20, "missing_value": 1e20},
        ),
        "rlut": xr.DataArray(
            np.random.uniform(150, 280, n).astype("float32"),
            dims=["time", "lat", "lon"],
            attrs={"units": "W m-2", "_FillValue": 1e20, "missing_value": 1e20},
        ),
    },
    coords={
        "time": np.arange(60, dtype="float64"),
        "lat": np.linspace(-90, 90, 73),
        "lon": np.linspace(0, 358.125, 192),
    },
)

# Run only the TOA check on the multi-variable dataset
toa_check = get_checks(["science.toa_energy_balance"])
report_toa = QCRunner(checks=toa_check).run(ds_rad, context={})
print(report_toa.summary())

## 8. Batch QC over a directory

For production workflows, iterate over CMORised files and collect all reports
into a single JSON summary.

In [ ]:
import glob
import json

runner = QCRunner()
reports = {}

# Replace with your output directory
# for path in glob.glob("output/**/*.nc", recursive=True):
#     ds = xr.open_dataset(path)
#     cmor_name = ds.attrs.get("variable_id", "unknown")
#     reports[path] = runner.run(ds, context={"cmor_name": cmor_name})
#     ds.close()

# --- Demo with our synthetic datasets ---
demo_datasets = {
    "tas_Amon_demo.nc": (ds, "tas", vocab),
    "pr_Amon_broken.nc": (ds_broken, "pr", vocab_pr),
}

for filename, (dataset, cmor_name, vocabulary) in demo_datasets.items():
    reports[filename] = runner.run(
        dataset,
        context={"cmor_name": cmor_name, "vocab": vocabulary},
    )

# Print summary table
print(f"{'File':<35} {'Status':<8} {'Pass':>5} {'Warn':>5} {'Fail':>5}")
print("-" * 65)
for filename, report in reports.items():
    print(
        f"{filename:<35} "
        f"{report.overall_status.value:<8} "
        f"{len(report.passed):>5} "
        f"{len(report.warnings):>5} "
        f"{len(report.failures):>5}"
    )

In [ ]:
# Persist to JSON
batch_results = {filename: report.to_dict() for filename, report in reports.items()}
print(json.dumps(batch_results, indent=2, default=str)[:1200], "...")

## 9. All registered checks

Print every check currently registered in the global registry.

In [ ]:
all_checks = get_checks()
print(f"{len(all_checks)} checks registered:\n")
for check in all_checks:
    print(f"  {check.name}")